# Лабораторная работа «Свёрточные нейронные сети»
## Часть 2. Oxford Pets и Transfer Learning

Классификация 37 пород кошек и собак с помощью предобученных на ImageNet сетей VGG-16, VGG-19 и ResNet-50. Свёрточная часть каждой сети заморожена, заново обучается только последний (классифицирующий) слой. Для лучшей сети считаются метрики и строятся карты GradCam.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from torchvision.datasets import OxfordIIITPet

from sklearn.metrics import confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)

### Данные

Оригинальный датасет Oxford-IIIT Pet скачивается с сервера авторов средствами torchvision. Используется официальное разбиение `trainval` / `test`. В файлах аннотаций для каждого изображения указаны номер породы и вид (1 - кошка, 2 - собака).

In [ ]:
OxfordIIITPet('data', download=True)   # скачивание и распаковка (около 800 МБ)
ROOT = 'data/oxford-iiit-pet'


def read_split(name):
    rows = []
    with open(os.path.join(ROOT, 'annotations', name + '.txt')) as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            img, cls, species = line.split()[:3]
            rows.append((os.path.join(ROOT, 'images', img + '.jpg'), int(cls) - 1, int(species) - 1))
    return rows   # (путь, порода, вид: 0 - кошка, 1 - собака)


train_rows, test_rows = read_split('trainval'), read_split('test')
NUM_CLASSES = len(set(r[1] for r in train_rows))

classes = [''] * NUM_CLASSES
class_is_dog = np.zeros(NUM_CLASSES, dtype=bool)
for path, c, s in train_rows:
    classes[c] = os.path.basename(path).rsplit('_', 1)[0]
    class_is_dog[c] = (s == 1)

y_train = torch.tensor([r[1] for r in train_rows])
y_test = torch.tensor([r[1] for r in test_rows])

print(f'train: {len(train_rows)} | test: {len(test_rows)} | классов: {NUM_CLASSES}')
print('кошек:', (~class_is_dog).sum(), '| собак:', class_is_dog.sum())
print(classes)

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

view_tf = T.Compose([T.Resize(256), T.CenterCrop(224)])                       # для показа
norm_tf = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])  # для сети
tf = T.Compose([view_tf, norm_tf])


class Pets(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return tf(Image.open(self.rows[i][0]).convert('RGB'))


fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, i in zip(axes.flat, np.random.choice(len(train_rows), 12, replace=False)):
    ax.imshow(view_tf(Image.open(train_rows[i][0]).convert('RGB')))
    ax.set_title(classes[train_rows[i][1]], fontsize=9)
    ax.axis('off')
plt.show()

### Предобученные сети

У каждой сети последний слой (1000 классов ImageNet) заменяется на `Identity`, все веса замораживаются. Выход такой сети - вектор признаков: 4096 чисел у VGG и 2048 у ResNet-50. Для GradCam запоминаем последний свёрточный слой.

In [ ]:
def build(name):
    if name == 'VGG-16':
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    elif name == 'VGG-19':
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
    else:
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    if name.startswith('VGG'):
        dim = m.classifier[6].in_features
        m.classifier[6] = nn.Identity()
        cam_layer = m.features[-2]     # ReLU после последней свёртки (перед MaxPool)
    else:
        dim = m.fc.in_features
        m.fc = nn.Identity()
        cam_layer = m.layer4[-1]       # последний свёрточный блок

    for p in m.parameters():
        p.requires_grad = False
    return m.eval().to(device), dim, cam_layer


nets = {}
for name in ['VGG-16', 'VGG-19', 'ResNet-50']:
    model, dim, cam_layer = build(name)
    nets[name] = {'model': model, 'dim': dim, 'cam_layer': cam_layer}
    print(name, '- размер вектора признаков:', dim)

### Извлечение признаков

Так как свёрточная часть заморожена, её выходы для каждой картинки не меняются в процессе обучения. Поэтому считаем их один раз (за один проход по данным сразу для трёх сетей), а затем обучаем только новый слой. Это эквивалентно обучению всей сети с замороженными весами, но во много раз быстрее.

In [ ]:
@torch.no_grad()
def extract_all(rows, desc):
    dl = DataLoader(Pets(rows), batch_size=64, shuffle=False, num_workers=2)
    feats = {name: [] for name in nets}
    for x in tqdm(dl, desc=desc):
        x = x.to(device)
        with torch.autocast(device_type=device, enabled=(device == 'cuda')):
            for name, net in nets.items():
                feats[name].append(net['model'](x).float().cpu())
    return {name: torch.cat(v) for name, v in feats.items()}


X_train = extract_all(train_rows, 'train')
X_test = extract_all(test_rows, 'test')
for name in nets:
    print(name, tuple(X_train[name].shape), tuple(X_test[name].shape))

### Обучение трёх классификаторов

Новая «голова» - Dropout и линейный слой на 37 классов.

In [ ]:
def train_head(Xtr, ytr, Xte, yte, epochs=30, lr=1e-3, wd=1e-3, bs=128):
    torch.manual_seed(SEED)
    head = nn.Sequential(nn.Dropout(0.5), nn.Linear(Xtr.shape[1], NUM_CLASSES)).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * ((len(Xtr) + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    Xtr, ytr, Xte = Xtr.to(device), ytr.to(device), Xte.to(device)

    hist = []
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(Xtr), bs):
            b = perm[i:i + bs]
            loss = F.cross_entropy(head(Xtr[b]), ytr[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
        head.eval()
        with torch.no_grad():
            logits = head(Xte).cpu()
        hist.append((logits.argmax(1) == yte).float().mean().item())
    return head, logits, hist


for name, net in nets.items():
    net['head'], net['logits'], net['hist'] = train_head(X_train[name], y_train, X_test[name], y_test)
    net['acc'] = net['hist'][-1]
    print(f'{name}: accuracy на тесте = {net["acc"]:.4f}')

plt.figure(figsize=(7, 4))
for name, net in nets.items():
    plt.plot(net['hist'], label=name)
plt.xlabel('эпоха')
plt.ylabel('accuracy на тесте')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
best = max(nets, key=lambda k: nets[k]['acc'])
print('Лучшая сеть:', best)

pd.DataFrame({'сеть': list(nets), 'accuracy на тесте': [round(nets[k]['acc'], 4) for k in nets]})

### Метрики лучшей сети

Точность «кошки против собак» считается так: суммируем вероятности всех собачьих пород и сравниваем с 0.5.

In [ ]:
probs = nets[best]['logits'].softmax(1)
pred = probs.argmax(1)


def topk_acc(k):
    return (probs.topk(k, dim=1).indices == y_test[:, None]).any(1).float().mean().item()


acc, top3, top5 = topk_acc(1), topk_acc(3), topk_acc(5)

is_dog = torch.tensor(class_is_dog)
true_dog = torch.tensor([r[2] == 1 for r in test_rows])
pred_dog = probs[:, is_dog].sum(1) > 0.5
acc_bin = (pred_dog == true_dog).float().mean().item()

print(f'Сеть: {best}')
print(f'Accuracy (top-1) по породам:   {acc:.4f}')
print(f'Top-3 accuracy:                {top3:.4f}')
print(f'Top-5 accuracy:                {top5:.4f}')
print(f'Accuracy кошки/собаки:         {acc_bin:.4f}')

pd.DataFrame(confusion_matrix(true_dog.numpy(), pred_dog.numpy(), labels=[False, True]),
             index=['истина: кошка', 'истина: собака'],
             columns=['предсказано: кошка', 'предсказано: собака'])

In [ ]:
n = NUM_CLASSES
cm = confusion_matrix(y_test.numpy(), pred.numpy(), labels=list(range(n)))

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticks(range(n))
ax.set_yticklabels(classes, fontsize=7)
ax.set_xlabel('Предсказанный класс')
ax.set_ylabel('Истинный класс')
ax.set_title('Confusion matrix: ' + best + ', тестовая выборка')
for i in range(n):
    for j in range(n):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Какие породы сеть путает чаще всего
off = cm.copy()
np.fill_diagonal(off, 0)
for k in np.argsort(off.ravel())[::-1][:10]:
    i, j = divmod(int(k), n)
    print(f'{classes[i]:>28s} -> {classes[j]:<28s} {off[i, j]}')

### GradCam

Для выбранного класса считаем градиент его выхода по картам признаков последнего свёрточного слоя, усредняем градиент по пространству (получаем вес каждого канала), складываем карты с этими весами и применяем ReLU. Результат растягиваем до размера изображения.

Ниже матрица 5x5: строки - изображения, столбцы - классы, для которых строится карта. На диагонали стоит «свой» класс изображения. В заголовке указана вероятность класса по мнению сети. Каждая карта нормирована на свой максимум, поэтому для «чужих» классов она показывает, какие области хоть как-то говорят в пользу этого класса, даже если вероятность близка к нулю.

In [ ]:
cam_model = nn.Sequential(nets[best]['model'], nets[best]['head']).eval()
cam_layer = nets[best]['cam_layer']


def grad_cam(model, layer, x, class_idx):
    store = {}

    def hook(module, inp, out):
        out.retain_grad()
        store['act'] = out

    handle = layer.register_forward_hook(hook)
    x = x.clone().requires_grad_(True)   # веса заморожены, поэтому граф строим от входа
    out = model(x)
    handle.remove()

    model.zero_grad()
    out[0, class_idx].backward()

    act, grad = store['act'], store['act'].grad
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * act).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)[0, 0]
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy(), out.softmax(1)[0, class_idx].item()

In [ ]:
cam_names = ['Abyssinian', 'Siamese', 'pug', 'samoyed', 'beagle']
cam_classes = [classes.index(name) for name in cam_names]

# для каждого класса берём первое тестовое изображение, которое сеть распознала верно
cam_images = []
for c in cam_classes:
    ids = [i for i, r in enumerate(test_rows) if r[1] == c]
    ok = [i for i in ids if pred[i].item() == c]
    cam_images.append((ok or ids)[0])

fig, axes = plt.subplots(5, 5, figsize=(15, 15.5))
for r, i in enumerate(cam_images):
    img = view_tf(Image.open(test_rows[i][0]).convert('RGB'))
    x = norm_tf(img)[None].to(device)
    for k, c in enumerate(cam_classes):
        cam, p = grad_cam(cam_model, cam_layer, x, c)
        ax = axes[r, k]
        ax.imshow(img)
        ax.imshow(cam, cmap='jet', alpha=0.45)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{classes[c]}: p = {p:.2f}', fontsize=10)
        if k == 0:
            ax.set_ylabel('фото: ' + classes[test_rows[i][1]], fontsize=11)
plt.suptitle('GradCam, ' + best + ' (строки - изображения, столбцы - классы)', fontsize=13)
plt.tight_layout()
plt.show()

## Итоги

In [ ]:
rows = [(name + ': accuracy по породам', nets[name]['acc']) for name in nets]
rows += [
    (best + ' (лучшая): top-3 accuracy', top3),
    (best + ' (лучшая): top-5 accuracy', top5),
    (best + ' (лучшая): accuracy кошки/собаки', acc_bin),
]
pd.DataFrame([(m, round(v, 4)) for m, v in rows], columns=['метрика', 'значение'])